# Step 3: Data Preprocessing

**Input:** `ncdb_2017_2018_2019_collisions.csv` (from notebook 01, NCDB 2017-2019)
**Output:**
- `ncdb_clean.csv`: cleaned collision table with a `split` column (train/test)
- `X_train.csv.gz`, `X_test.csv.gz`, `y_train.csv`, `y_test.csv`: encoded and scaled, all features
- `X_train_selected.csv.gz`, `X_test_selected.csv.gz`: same, only the selected features

**Sections**
0. Settings (all preprocessing choices in one place)
1. Load and filter
2. Feature engineering (hour periods)
3. Outliers
4. Missing values
5. Train/test split (stratified)
6. Rare categories
7. Encoding and scaling
8. Feature selection (mutual information)
9. Dimensionality reduction check (PCA)
10. Class imbalance (class weights)
11. Save
12. Summary

## 0. Settings

In [ ]:
DATA_FILE = "ncdb_2017_2018_2019_collisions.csv"

INTERSECTION_ONLY = True      # True = keep only intersection collisions (matches project title)
VEH_CAP = 6                   # cap vehicle count: 6 means "6 or more"
MISSING_DROP_PCT = 5.0        # feature missing < this %: drop those rows; otherwise "Unknown" category
RARE_PCT = 1.0                # categories under this % of training rows are merged into "Other"
TEST_SIZE = 0.2               # 80/20 split
N_KEEP = 10                   # number of features kept by feature selection (out of 13)
PCA_VARIANCE = 0.90           # PCA check: components needed to explain this much variance
RANDOM_STATE = 42

## 1. Load and filter

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

if not os.path.exists(DATA_FILE):
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]

CODE_COLS = ["C_SEV", "C_CONF", "C_RCFG", "C_WTHR", "C_RSUR", "C_RALN", "C_TRAF", "C_CASE"]
col = pd.read_csv(DATA_FILE, dtype={c: str for c in CODE_COLS})
log = []   # keeps track of how many rows each step removes
log.append(("Loaded", len(col)))
print("Loaded:", col.shape)

In [ ]:
# Drop collisions with no class (0 known casualties): there is no answer to learn from
col = col[col["Casualty_Class"].notna()].copy()
col["Casualty_Class"] = col["Casualty_Class"].astype(str)
log.append(("Dropped rows with no Casualty_Class", len(col)))

if INTERSECTION_ONLY:
    col = col[col["Is_Intersection"] == 1].copy()
    log.append(("Kept intersection collisions only", len(col)))

print(pd.DataFrame(log, columns=["step", "rows"]))

## 2. Feature engineering

Hour is turned into 4 time periods. As a plain number, the model would think 23:00 and 00:00 are far apart;
as 24 categories, it would add many columns. Day of week and month are treated as categories.

Readable labels replace the NCDB codes so feature names make sense later (check them against the data dictionary).

In [ ]:
LABELS = {
    "C_WTHR": {"1": "Clear", "2": "Overcast", "3": "Rain", "4": "Snow",
               "5": "Freezing rain/hail", "6": "Low visibility", "7": "Strong wind", "Q": "Other"},
    "C_RSUR": {"1": "Dry", "2": "Wet", "3": "Snow", "4": "Slush", "5": "Icy",
               "6": "Sand/gravel", "7": "Muddy", "8": "Oil", "9": "Flooded", "Q": "Other"},
    "C_RALN": {"1": "Straight level", "2": "Straight gradient", "3": "Curved level",
               "4": "Curved gradient", "5": "Top of hill", "6": "Bottom of hill", "Q": "Other"},
    "C_RCFG": {"01": "Non-intersection", "02": "Intersection", "03": "Driveway/parking",
               "04": "Railroad crossing", "05": "Bridge/overpass", "06": "Tunnel",
               "07": "Passing lane", "08": "Ramp", "09": "Traffic circle",
               "10": "Express lane", "11": "Collector lane", "12": "Transfer lane", "QQ": "Other"},
    "C_TRAF": {"01": "Signals working", "02": "Signals flashing", "03": "Stop sign",
               "04": "Yield sign", "05": "Warning sign", "06": "Ped crosswalk",
               "07": "Police officer", "08": "School guard", "09": "School crossing",
               "10": "Reduced speed zone", "11": "No passing zone", "12": "Road markings",
               "13": "School bus", "15": "Rail signals/gates", "16": "Rail signs only",
               "17": "Not specified", "18": "No control", "QQ": "Other"},
    "C_CONF": {"01": "Hit moving object", "02": "Hit stationary object", "03": "Ran off left",
               "04": "Ran off right", "05": "Rollover", "06": "Other single vehicle",
               "21": "Rear-end", "22": "Side swipe", "23": "Left turn conflict same dir",
               "24": "Right turn conflict same dir", "25": "Other same direction",
               "31": "Head-on", "32": "Approaching side swipe", "33": "Left turn across traffic",
               "34": "Right turn", "35": "Right angle", "36": "Other different direction",
               "41": "Hit parked vehicle", "QQ": "Other"},
}
DAYS = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}

for c, mapping in LABELS.items():
    col[c] = col[c].map(lambda v: mapping.get(v, v) if pd.notna(v) else np.nan)

def hour_period(h):
    if pd.isna(h): return np.nan
    if h < 6:  return "Night (0-5)"
    if h < 12: return "Morning (6-11)"
    if h < 18: return "Afternoon (12-17)"
    return "Evening (18-23)"

col["Hour_Period"] = col["C_HOUR"].apply(hour_period)
col["Weekday"] = col["C_WDAY"].map(DAYS)
col["Month"] = col["C_MNTH"].map(lambda m: f"M{int(m):02d}" if pd.notna(m) else np.nan)

col["Hour_Period"].value_counts(dropna=False)

In [ ]:
CAT_FEATURES = ["C_RCFG", "C_TRAF", "C_WTHR", "C_RSUR", "C_RALN", "C_CONF",
                "Hour_Period", "Weekday", "Month"]
NUM_FEATURES = ["C_VEHS"]
BIN_FEATURES = ["Heavy_Vehicle_Present", "Cyclist_Present", "Safety_Device_Not_Used"]
FEATURES = CAT_FEATURES + NUM_FEATURES + BIN_FEATURES   # 13 features
TARGET = "Casualty_Class"
CLASS_ORDER = ["1", "2", "3+"]

# Leakage columns (built from the target) are never used as inputs
LEAKAGE = ["Total_Injuries", "Total_Fatalities", "Total_Casualties", "Fatal_Collision", "C_SEV"]
print(len(FEATURES), "features")

## 3. Outliers

Vehicle count is the only numeric feature where outliers matter. Large pileups are real events and are
often 3+ casualty collisions (the smallest class), so they are **capped** at `VEH_CAP` instead of removed.
Further outlier analysis (Isolation Forest) is planned for the final project.

In [ ]:
q1, q3 = col["C_VEHS"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
above_fence = col["C_VEHS"] > fence
print(f"IQR upper fence: {fence:.1f} vehicles -> {above_fence.sum():,} collisions above it")
print("Class mix above fence (%):")
print((col.loc[above_fence, TARGET].value_counts(normalize=True) * 100).round(1))

capped = (col["C_VEHS"] > VEH_CAP).sum()
col["C_VEHS"] = col["C_VEHS"].clip(upper=VEH_CAP)
print(f"\nCapped {capped:,} collisions at {VEH_CAP} vehicles")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.boxplot(x=col["C_VEHS"], ax=axes[0], color="lightsteelblue")
axes[0].set_title(f"Vehicles after capping at {VEH_CAP}")
sns.boxplot(data=col, x=TARGET, y="C_VEHS", order=CLASS_ORDER, ax=axes[1], color="lightsteelblue")
axes[1].set_title("Vehicles by Casualty_Class (after capping)")
plt.tight_layout(); plt.show()

## 4. Missing values

Rule applied to every feature:
- **Missing < `MISSING_DROP_PCT`%** -> drop those rows (small loss, no guessing)
- **Missing >= `MISSING_DROP_PCT`%** -> categorical: new **"Unknown"** category; binary: becomes a 3-level category (Yes/No/Unknown); numeric: median (fitted on training data only)

Keeping "Unknown" avoids losing many rows, and in the NCDB missing often means a province does not report
that field, so it can carry information. Dropping rows before the split is fine; median filling happens after the split.

In [ ]:
miss_pct = col[FEATURES].isna().mean() * 100
decision = pd.DataFrame({"pct_missing": miss_pct.round(2)})
# Use the exact %, not the rounded one, so even a handful of missing values gets handled
decision["action"] = np.where(miss_pct == 0, "none",
                      np.where(miss_pct < MISSING_DROP_PCT, "drop rows", "Unknown / median"))
decision.sort_values("pct_missing", ascending=False)

In [ ]:
drop_cols = decision.index[decision["action"] == "drop rows"].tolist()
before = len(col)
col = col.dropna(subset=drop_cols)
log.append((f"Dropped rows missing in: {', '.join(drop_cols) or 'none'}", len(col)))
print(f"Dropped {before - len(col):,} rows ({(before - len(col)) / before * 100:.2f}%)")

unknown_cols = decision.index[decision["action"] == "Unknown / median"].tolist()
for c in unknown_cols:
    if c in CAT_FEATURES:
        col[c] = col[c].fillna("Unknown")
    elif c in BIN_FEATURES:
        col[c] = col[c].map({1: "Yes", 0: "No", 1.0: "Yes", 0.0: "No"}).fillna("Unknown")
        BIN_FEATURES.remove(c); CAT_FEATURES.append(c)   # now treated as a category
# Numeric features with many missing values are median-filled after the split (section 5)
MEDIAN_FILL = [c for c in unknown_cols if c in NUM_FEATURES]

print("Became 'Unknown' category:", [c for c in unknown_cols if c not in MEDIAN_FILL] or "none")
print("Median fill after split:", MEDIAN_FILL or "none")
print("Rows now:", f"{len(col):,}")

## 5. Train/test split (stratified)

Stratified so each set keeps the same class mix (important because 3+ is rare).
Everything learned from data (rare categories, medians, scaling, encoding, feature selection) is fitted on
the **training set only**, so no test information leaks into training.

In [ ]:
train_df, test_df = train_test_split(col, test_size=TEST_SIZE, stratify=col[TARGET],
                                     random_state=RANDOM_STATE)
train_df, test_df = train_df.copy(), test_df.copy()

for c in MEDIAN_FILL:
    med = train_df[c].median()
    train_df[c] = train_df[c].fillna(med); test_df[c] = test_df[c].fillna(med)

split_mix = pd.DataFrame({
    "train %": train_df[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
    "test %": test_df[TARGET].value_counts(normalize=True).reindex(CLASS_ORDER) * 100,
}).round(2)
print(f"Train: {len(train_df):,}   Test: {len(test_df):,}")
split_mix

## 6. Rare categories

Categories with fewer than `RARE_PCT`% of training rows give the model very few examples, and they add extra
columns after one-hot encoding. They are merged into **"Other"** (decided on training data, applied to both sets).

In [ ]:
rare_report = []
for c in CAT_FEATURES:
    freq = train_df[c].value_counts(normalize=True) * 100
    rare = freq[freq < RARE_PCT].index.tolist()
    if rare:
        train_df[c] = train_df[c].where(~train_df[c].isin(rare), "Other")
    keep = set(train_df[c].unique())
    test_df[c] = test_df[c].where(test_df[c].isin(keep), "Other")
    rare_report.append({"feature": c, "categories before": len(freq),
                        "merged into Other": len(rare), "categories after": train_df[c].nunique()})
pd.DataFrame(rare_report)

## 7. Encoding and scaling

- **Categorical** -> one-hot encoding (one 0/1 column per category). No order is implied, which fits nominal
  codes like weather or collision type.
- **Numeric** (vehicle count) -> StandardScaler (mean 0, standard deviation 1), fitted on training data.
  Mainly matters for logistic regression and k-means; tree models are not affected by scale.
- **Binary** features are already 0/1 and are passed through unchanged.

In [ ]:
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES),
    ("bin", "passthrough", BIN_FEATURES),
], verbose_feature_names_out=False)

X_train = pd.DataFrame(pre.fit_transform(train_df[FEATURES]),
                       columns=pre.get_feature_names_out(), index=train_df.index)
X_test = pd.DataFrame(pre.transform(test_df[FEATURES]),
                      columns=pre.get_feature_names_out(), index=test_df.index)
y_train, y_test = train_df[TARGET], test_df[TARGET]
assert not X_train.isna().any().any() and not X_test.isna().any().any(), "NaN left after preprocessing"

print(f"Columns before encoding: {len(FEATURES)}   after encoding: {X_train.shape[1]}")
print("Vehicle count after scaling (train): mean "
      f"{X_train['C_VEHS'].mean():.2f}, std {X_train['C_VEHS'].std():.2f}")
X_train.head()

## 8. Feature selection: mutual information

Mutual information (MI) measures how much knowing a feature reduces uncertainty about the casualty class
(0 = no information). It is a **filter method**, independent of any classification model, and works for
categorical features and non-linear relationships. It is computed per original feature on the training set.
The top `N_KEEP` features are kept; the final project will compare models with and without this selection.

In [ ]:
enc = OrdinalEncoder()
X_mi = enc.fit_transform(train_df[FEATURES].astype(str))
mi = mutual_info_classif(X_mi, y_train, discrete_features=True, random_state=RANDOM_STATE)
mi = pd.Series(mi, index=FEATURES).sort_values(ascending=False)

selected = mi.head(N_KEEP).index.tolist()
dropped = mi.index[N_KEEP:].tolist()

colors = ["#4C72B0" if f in selected else "#BBBBBB" for f in mi.index]
ax = mi.sort_values().plot.barh(figsize=(7, 5), color=colors[::-1])
ax.set_xlabel("Mutual information with Casualty_Class")
ax.set_title(f"Feature selection: top {N_KEEP} kept (blue), dropped (grey)")
plt.tight_layout(); plt.show()

print("Selected:", selected)
print("Dropped: ", dropped)
mi.round(5).to_frame("MI")

In [ ]:
# Encoded columns that belong to the selected features
def belongs(colname, feats):
    return any(colname == f or colname.startswith(f + "_") for f in feats)

sel_cols = [c for c in X_train.columns if belongs(c, selected)]
X_train_sel, X_test_sel = X_train[sel_cols], X_test[sel_cols]
print(f"Encoded columns: all features {X_train.shape[1]} -> selected features {X_train_sel.shape[1]}")

## 9. Dimensionality reduction check (PCA)

PCA is only needed if the data is high-dimensional. This checks how many components are needed to keep
`PCA_VARIANCE` of the variance. PCA is **not applied** to the saved data by default because it turns features
into combinations that are hard to interpret (we want to know which factors matter). It will be used in the
final project to visualize clusters in 2D.

In [ ]:
pca = PCA(random_state=RANDOM_STATE).fit(X_train)
cum = np.cumsum(pca.explained_variance_ratio_)
n_needed = int(np.argmax(cum >= PCA_VARIANCE) + 1)

ax = plt.figure(figsize=(7, 3.5)).gca()
ax.plot(range(1, len(cum) + 1), cum, marker=".")
ax.axhline(PCA_VARIANCE, color="red", ls="--", lw=1)
ax.axvline(n_needed, color="red", ls="--", lw=1)
ax.set_xlabel("Number of components"); ax.set_ylabel("Cumulative explained variance")
ax.set_title("PCA explained variance (training set)")
plt.tight_layout(); plt.show()

print(f"Encoded columns: {X_train.shape[1]}")
print(f"Components for {PCA_VARIANCE:.0%} variance: {n_needed} "
      f"({n_needed / X_train.shape[1]:.0%} of columns)")

## 10. Class imbalance

The 3+ class is rare. Plan: **stratified split** (done above) + **class weights** in the models, so mistakes on
rare classes cost more. Resampling such as SMOTE can be compared in the final project, applied to the training
set only. The weights below are computed now so they can be reused.

In [ ]:
weights = compute_class_weight("balanced", classes=np.array(CLASS_ORDER), y=y_train)
class_weights = {c: round(float(w), 3) for c, w in zip(CLASS_ORDER, weights)}
print("Training class counts:", y_train.value_counts().reindex(CLASS_ORDER).to_dict())
print("Balanced class weights:", class_weights)

## 11. Save

In [ ]:
clean = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")])
clean_cols = ["C_CASE", "C_YEAR"] + FEATURES + [TARGET, "Is_Intersection"] + \
             [c for c in LEAKAGE if c in clean.columns] + ["split"]
clean[clean_cols].to_csv("ncdb_clean.csv", index=False)

X_train.to_csv("X_train.csv.gz", index=False); X_test.to_csv("X_test.csv.gz", index=False)
X_train_sel.to_csv("X_train_selected.csv.gz", index=False)
X_test_sel.to_csv("X_test_selected.csv.gz", index=False)
y_train.to_csv("y_train.csv", index=False); y_test.to_csv("y_test.csv", index=False)

out_files = ["ncdb_clean.csv", "X_train.csv.gz", "X_test.csv.gz", "X_train_selected.csv.gz",
             "X_test_selected.csv.gz", "y_train.csv", "y_test.csv"]
for f in out_files:
    print(f"{f:28s} {os.path.getsize(f) / 1e6:6.2f} MB")

try:
    from google.colab import files
    for f in out_files:
        files.download(f)
except ImportError:
    pass

## 12. Summary

In [ ]:
print("Row flow:")
print(pd.DataFrame(log, columns=["step", "rows"]).to_string(index=False))
print(f"\nTrain / test: {len(train_df):,} / {len(test_df):,} (stratified {1-TEST_SIZE:.0%}/{TEST_SIZE:.0%})")
print(f"Vehicle count capped at {VEH_CAP}; standardized")
print("'Unknown' category used for:", [c for c in unknown_cols if c not in MEDIAN_FILL] or "none")
print(f"Rare categories (<{RARE_PCT}%) merged into 'Other'")
print(f"One-hot encoded columns: {X_train.shape[1]} (selected: {X_train_sel.shape[1]})")
print(f"Selected features ({N_KEEP}):", selected)
print(f"PCA: {n_needed} components for {PCA_VARIANCE:.0%} variance (not applied)")
print("Class weights:", class_weights)